# Lab 04 (Colab): What perplexity tells you (and what it doesn't)

**Chapter 3, "Evaluation Methodology"**, section *Understanding Language Modeling Metrics* (cross entropy, bits-per-byte, perplexity) and its subsection *Perplexity Interpretation and Use Cases*
**Environment:** Google Colab, free **T4 GPU** | **Time:** about 45 minutes (the cluster notebook is the bigger part of this lab)

A language model gives every possible next token a probability. Chapter 3 turns that into a few scores: **cross-entropy** (how surprised the model is on average), **perplexity** (the same thing as "how many equally likely choices the model was torn between"), and **bits-per-byte** (a version you can compare across tokenizers). The code that computes them is given to you. Your job is to **predict, run, and interpret**: which texts does a model find predictable, how do the model and its tokenizer change the numbers, and can an IT team use perplexity to flag garbled or abnormal tickets automatically?

**Before you start**
- Runtime > Change runtime type > **T4 GPU**. Then Runtime > Run all is safe: every TODO has a placeholder.
- This notebook downloads three small models (about 3.5 GB in total; a few minutes on Colab). It does not use the course cluster (Colab can't reach it).

## Words you'll meet in this lab

You don't need to memorize these. Come back here if a word stops you.

| Term | What it means | Why it matters |
|---|---|---|
| **Token, tokenizer** | The tokenizer splits text into tokens and gives each an ID number (from Lab 01). | Scores per token depend on how the text was split. |
| **Byte, UTF-8** | UTF-8 is the standard way computers store text: one byte per plain English letter, more for accented letters and emoji. | Counting bytes gives a measure that doesn't depend on the tokenizer. |
| **Cross-entropy** | How surprised the model is, on average, by each actual next token. Measured in *nats* (natural log) or *bits* (log base 2). | Lower means the model predicts the text better. |
| **Perplexity** | Cross-entropy turned into "how many equally likely tokens was the model choosing between, on average". | The classic score for a language model; lower is better. |
| **Bits per byte (BPB)** | Total bits of surprise divided by the number of bytes of text. | Unlike perplexity, it stays fair when the tokenizer, or the kind of text, changes how many tokens the text is cut into. |
| **Base vs post-trained model** | Base: only trained to continue text. Post-trained: further trained to follow instructions (from Lab 03). | B2 shows how post-training changes perplexity. |
| **Data contamination** | When a test text was already in the model's training data, so the model has partly memorized it. | It makes scores on public benchmarks look better than they are. |

## Setup

The first cell installs the course helper package. The second loads the libraries and checks for a GPU. The third defines the helpers you'll use all lab, including **`lm_metrics()`**, and loads `Qwen/Qwen3-0.6B` in 32-bit precision (`float32`), so the math is exact. You don't need to change any of this code.

In [ ]:
!pip -q install "itec632 @ git+https://github.com/tyler-schunk/itec632-course#subdirectory=image"

In [ ]:
import gc, math, random, string, time

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from itec632.grading import LabChecker

ck = LabChecker("lab04", part="colab")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(no GPU: Runtime > Change runtime type > T4 GPU)")

In [ ]:
def load_model(name):
    """Load a tokenizer and causal LM in float32 (exact math; these models are small enough for a T4)."""
    tok = AutoTokenizer.from_pretrained(name)
    try:
        model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)
    except TypeError:                       # older transformers used torch_dtype=
        model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float32)
    return tok, model.to(DEVICE).eval()


def free_gpu():
    """Release GPU memory after you `del` a model."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def start_token_id(tok):
    """A document-start token, put in front of every text so that EVERY token of the text gets predicted."""
    start = tok.convert_tokens_to_ids("<|endoftext|>")
    if start is None or start == tok.unk_token_id:
        start = tok.bos_token_id if tok.bos_token_id is not None else tok.eos_token_id
    return start


def lm_metrics(tok, model, text, context=""):
    """How predictable `text` is for `model`.

    Returns tokens, bytes, cross-entropy (nats per token), bits per token, perplexity, and bits per byte (bpb).
    If `context` is given, the model reads it first but only `text` is scored (used in B3).
    """
    ctx_ids = tok(context, add_special_tokens=False)["input_ids"] if context else []
    txt_ids = tok(text, add_special_tokens=False)["input_ids"]
    ids = torch.tensor([[start_token_id(tok)] + ctx_ids + txt_ids], device=DEVICE)
    with torch.no_grad():
        logits = model(ids).logits[0, :-1].float()          # the scores at position i predict token i+1
    logprobs = torch.log_softmax(logits, dim=-1)            # scores -> log-probabilities
    surprise = -logprobs.gather(1, ids[0, 1:, None]).squeeze(1)   # -log p(actual next token), in nats
    surprise = surprise[-len(txt_ids):]                     # score only the text, not the context
    n_bytes = len(text.encode("utf-8"))
    ce_nats = surprise.mean().item()
    return {"tokens": len(txt_ids), "bytes": n_bytes, "ce_nats": ce_nats,
            "bits_per_token": ce_nats / math.log(2),
            "perplexity": math.exp(ce_nats),
            "bpb": surprise.sum().item() / math.log(2) / n_bytes}


MAIN = "Qwen/Qwen3-0.6B"
tok, model = load_model(MAIN)
print(MAIN, "loaded;", round(sum(p.numel() for p in model.parameters()) / 1e6), "M parameters; vocab", len(tok))

# Part A - Guided practice

## A1. Three numbers for one text

Our main text is the opening of Larkspur's current runbook **RB-PRT-001 "Printer shows offline"** (v2). `lm_metrics()` runs the model over it once and, for every token, looks up the probability the model gave to the token that actually came next. From those probabilities it computes:

- **Cross-entropy** (`ce_nats`): the average surprise per token. Near 0 means the model was almost always sure and right. Dividing by ln 2 (about 0.693) converts nats to bits (`bits_per_token`).
- **Perplexity**: e to the power of the cross-entropy. The chapter's reading: if the model were picking uniformly among this many tokens, it would be as uncertain as it actually is. Lower is better.
- **Bits per byte** (`bpb`): the total bits of surprise for the whole text divided by its length in bytes. Perplexity is *per token*, so it changes when a different tokenizer cuts the text differently; BPB is per byte of text, so it doesn't.

**TODO:** compute how many bytes of text each token covers on average (use the `tokens` and `bytes` the function returned).

In [ ]:
PASSAGE = 'Use when a user reports a printer shows **Offline** in Windows, jobs sit in the queue and never print, or "Error - Printing" won\'t clear. Applies to network HP and Brother printers/MFPs at client sites, printed to directly from Windows 10/11 PCs via a Standard TCP/IP port. If the printer is shared from a client print server, see the print server note in step 9.\n\nWork the steps **in order**. Most "offline" printers are either a network/IP problem or a stuck spool, and clearing the spool before confirming the printer is reachable just wastes time.\n\n\n\n## Steps\n\n### 1. Check power and network first\n\n1. Ask the user (or a coworker near the printer) to look at the printer:\n   - Is it powered on and awake? Press a button to wake it from sleep.\n   - Any error on the front panel (paper jam, toner, "door open", "IP conflict")? Clear physical errors first; jams and empty trays often show as "offline" or "error" in Windows.\n   - Is the network cable plugged in with link lights? On Wi-Fi printers, is the Wi-Fi indicator solid?\n2. **Find the printer\'s current IP** from its front panel (HP: Settings/Network or the network icon; Brother: Menu > Network > Wired LAN or WLAN > TCP/IP). Compare with the IP in the client\'s Atlas Docs printer list and the IP on the Windows port:\n   - On the user\'s PC: Settings > Bluetooth & devices > Printers & scanners > the printer > **Printer properties** > **Ports** tab, note the checked port and its address (Configure Port).'

m = lm_metrics(tok, model, PASSAGE)
print({k: round(v, 3) for k, v in m.items()})

In [ ]:
bytes_per_token = None     # TODO: average number of bytes per token for the runbook passage (both counts are in `m`)
print("bytes per token:", bytes_per_token)

In [ ]:
ck.check("A1_bytes_per_token", bytes_per_token is not None and abs(bytes_per_token - m["bytes"] / m["tokens"]) <= 0.01 * m["bytes"] / m["tokens"]
         and 2 < bytes_per_token < 8, points=2, hint="Divide the passage's byte count by its token count (both are keys of m). English is usually 3-5 bytes per token.")
ck.record("A1_runbook_ppl_qwen3_0.6b", m["perplexity"])
ck.record("A1_runbook_bpb_qwen3_0.6b", m["bpb"])

## A2. Reading a perplexity

*Perplexity Interpretation and Use Cases* gives a way to read the number: if every token were equally likely, a perplexity of 3 would mean the model has a 1 in 3 chance of guessing the next token. Its vocabulary has well over 100,000 tokens, so a small perplexity means the model is doing far better than chance.

**TODO:** turn your runbook perplexity from A1 into that "chance of guessing the next token" (a number between 0 and 1).

In [ ]:
guess_chance = None     # TODO: the chance (0-1) that matches your perplexity, following the chapter's "perplexity of 3" example
if guess_chance is not None:
    print(f"perplexity {m['perplexity']:.2f} -> about a 1 in {m['perplexity']:.1f} chance ({guess_chance:.1%}) per token")

In [ ]:
ck.check("A2_guess_chance", guess_chance is not None and abs(guess_chance - 1 / m["perplexity"]) <= 0.01 / m["perplexity"], points=1,
         hint="Perplexity 3 means a 1 in 3 chance. Do the same with your own perplexity from m.")

# Part B - Applied lab

## B1. Predict, then measure: structured, scrambled, random, famous, and yours

*Perplexity Interpretation and Use Cases* gives rules of thumb: more structured data is more predictable (lower perplexity), and perplexity is lowest for text the model saw and memorized during training. Five texts:

1. `runbook`: the Larkspur runbook passage,
2. `shuffled`: the same passage with its words shuffled (same words, no structure),
3. `random_chars`: random characters of the same length,
4. `pride_prejudice`: the opening of *Pride and Prejudice* (1813, public domain, and almost certainly in every model's training data),
5. `my_text`: **your own paragraph** (at least 300 characters). Something you wrote yourself and never posted online is most interesting.

**TODO (before you run the table):** write your own paragraph, and list the five names in the order you **predict**, from most predictable to least. The table ranks them by bits per byte (lowest first) and also shows the perplexity order, which may surprise you. Your prediction is not graded on being right; it is there so you notice what surprised you.

In [ ]:
rng = random.Random(632)
words = PASSAGE.split()
SHUFFLED = " ".join(rng.sample(words, len(words)))
RANDOM_CHARS = "".join(rng.choice(string.ascii_letters + string.digits + string.punctuation + " ") for _ in range(len(PASSAGE)))
PRIDE = 'It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife.\n\nHowever little known the feelings or views of such a man may be on his first entering a neighbourhood, this truth is so well fixed in the minds of the surrounding families, that he is considered the rightful property of some one or other of their daughters.\n\n"My dear Mr. Bennet," said his lady to him one day, "have you heard that Netherfield Park is let at last?"\n\nMr. Bennet replied that he had not.\n\n"But it is," returned she; "for Mrs. Long has just been here, and she told me all about it."\n\nMr. Bennet made no answer.\n\n"Do you not want to know who has taken it?" cried his wife impatiently.\n\n"You want to tell me, and I have no objection to hearing it."\n\nThis was invitation enough.'
print(SHUFFLED[:200], "...")
print(RANDOM_CHARS[:200], "...")

In [ ]:
MY_TEXT = ""          # TODO: your own paragraph (at least 300 characters)
MY_PREDICTION = []    # TODO: the five names above, ordered from most predictable (lowest bits per byte) to least

In [ ]:
TEXTS = {"runbook": PASSAGE, "shuffled": SHUFFLED, "random_chars": RANDOM_CHARS, "pride_prejudice": PRIDE}
if len(MY_TEXT) >= 300:
    TEXTS["my_text"] = MY_TEXT
b1 = pd.DataFrame([{"text": name, **lm_metrics(tok, model, text)} for name, text in TEXTS.items()]).set_index("text")
display(b1.round(3))
actual = b1["bpb"].sort_values().index.tolist()
print("Actual order (lowest bits per byte first):", actual)
print("Your prediction:                          ", MY_PREDICTION)
print("Order by perplexity instead:              ", b1["perplexity"].sort_values().index.tolist())
print("For comparison: uniformly random characters from this 95-symbol alphabet carry log2(95) =", round(math.log2(95), 2), "bits per byte.")

**Reading the table:** one row per text. `tokens` and `bytes` are the text's length, `ce_nats` its cross-entropy, `bits_per_token` the same in bits, `perplexity` the average number of equally likely choices, and `bpb` bits per byte. For all of them, lower means the model found the text more predictable.

Look at `random_chars`: it has about the same number of bytes as the runbook but roughly three times as many tokens, because the tokenizer has no long tokens for random junk and cuts it into tiny pieces. Perplexity is an average *per token*, and each tiny piece is a smaller guess, so the per-token number can make random characters look *better* than shuffled words. Bits per byte measures the surprise for the whole text against its length, so it doesn't get fooled. Keep this in mind for B4.

In [ ]:
NAMES = {"runbook", "shuffled", "random_chars", "pride_prejudice", "my_text"}
ppl = b1["perplexity"]
ck.check("B1_prediction", isinstance(MY_PREDICTION, list) and len(MY_PREDICTION) == 5 and set(MY_PREDICTION) == NAMES,
         points=1, hint="MY_PREDICTION must list each of the five text names exactly once (spelled as in the list above).")
ck.check("B1_my_text", "my_text" in TEXTS, points=1, hint="Set MY_TEXT to a paragraph of at least 300 characters, then re-run the table cell.")
ck.check("B1_table", ppl.notna().all() and len(b1) >= 4, points=1, hint="The table needs a perplexity for every text. Re-run the setup cells.")
bpb = b1["bpb"]
ck.check("B1_structure_ordering", bpb["random_chars"] > bpb["shuffled"] > bpb["runbook"], points=2,
         hint="Expected bits per byte: random_chars > shuffled > runbook. If not, re-run the cell that builds SHUFFLED and RANDOM_CHARS, then the table.")
ck.check("B1_memorized_text_is_easy", ppl["pride_prejudice"] < ppl["shuffled"], points=1,
         hint="The famous book passage should be far more predictable than shuffled words. Re-run the table cell.")
if len(MY_PREDICTION) == len(actual):
    ck.record("B1_prediction_positions_right", sum(p == a for p, a in zip(MY_PREDICTION, actual)))
for name in b1.index:
    ck.record(f"B1_ppl_{name}", float(b1.loc[name, "perplexity"]))
    ck.record(f"B1_bpb_{name}", float(b1.loc[name, "bpb"]))

## B2. Same text, different model and tokenizer

Perplexity is not a property of the text alone. You'll compare three small models on the runbook and on *Pride and Prejudice*:

| Model | What's different |
|---|---|
| `Qwen/Qwen3-0.6B` | the model you've been using: **post-trained** (instruction tuning, preference tuning) |
| `Qwen/Qwen3-0.6B-Base` | same size and tokenizer, pre-trained only |
| `HuggingFaceTB/SmolLM2-360M` | different family and a **different tokenizer** (about 49k tokens in its vocabulary vs Qwen's 151k) |

**TODO (before you run it):** predict whether the post-trained Qwen3-0.6B will have a **higher** or **lower** perplexity than its Base version. The chapter has an opinion on this; check your guess against it after the run.

In [ ]:
PREDICT_POST_TRAINING = ""     # TODO: "higher" or "lower" (post-trained Qwen3-0.6B compared with Qwen3-0.6B-Base)

In [ ]:
MODELS_B2 = ["Qwen/Qwen3-0.6B-Base", "HuggingFaceTB/SmolLM2-360M"]   # optional, not needed: add "Qwen/Qwen3-1.7B" (bigger; about 4 more minutes)
B2_TEXTS = {"runbook": PASSAGE, "pride_prejudice": PRIDE}
b2_rows = [{"model": MAIN, "text": t, **lm_metrics(tok, model, x)} for t, x in B2_TEXTS.items()]   # the model already loaded
for name in MODELS_B2:                    # about 3 minutes, mostly downloads
    t0 = time.time()
    tok_i, model_i = load_model(name)
    for text_name, text in B2_TEXTS.items():
        b2_rows.append({"model": name, "text": text_name, **lm_metrics(tok_i, model_i, text)})
    print(f"{name}: done in {time.time() - t0:.0f}s")
    del model_i
    free_gpu()
b2 = pd.DataFrame(b2_rows)
display(b2.pivot(index="model", columns="text", values=["tokens", "perplexity", "bpb"]).round(3))

**Reading the table:** one row per model. The columns come in three groups (`tokens`, `perplexity`, `bpb`), each split into the two texts. Compare `tokens` first: models that share a tokenizer split the text into the same number of tokens. Then compare `bpb`, the only column that is fair across different tokenizers.

**TODO:** copy the full name (as shown in the table) of the model with the **lowest bpb on the runbook**.

In [ ]:
LOWEST_BPB_MODEL = ""     # TODO: model name with the lowest runbook bpb, read from the table

In [ ]:
all_b2 = [MAIN] + MODELS_B2
run = b2[b2["text"] == "runbook"].set_index("model")
ok_table = len(b2) == 2 * len(all_b2) and len(b2) >= 6 and b2["perplexity"].notna().all()
ck.check("B2_prediction", PREDICT_POST_TRAINING in ("higher", "lower"), points=1,
         hint='Set PREDICT_POST_TRAINING to "higher" or "lower" (lowercase).')
ck.check("B2_table", ok_table, points=2, hint="You need every model x 2 texts in the table. Re-run the B2 cell (Runtime > Restart and run all if you ran out of memory).")
tok_effect = False
if ok_table:
    qwen_tokens = run.loc[[m_ for m_ in all_b2 if m_.startswith("Qwen")], "tokens"]
    tok_effect = ("HuggingFaceTB/SmolLM2-360M" in run.index and qwen_tokens.nunique() == 1
                  and run.loc["HuggingFaceTB/SmolLM2-360M", "tokens"] != qwen_tokens.iloc[0])
ck.check("B2_tokenizer_effect", tok_effect, points=1,
         hint="The Qwen models share a tokenizer (same token count); SmolLM2 should count differently. Re-run the B2 cell.")
ck.check("B2_lowest_bpb", ok_table and LOWEST_BPB_MODEL == run["bpb"].idxmin(), points=1,
         hint="Look at the runbook column of the bpb group and copy that model's full name exactly, including the part before the slash.")
if ok_table:
    for mname in all_b2:
        short = mname.split("/")[-1]
        ck.record(f"B2_runbook_ppl_{short}", float(run.loc[mname, "perplexity"]))
        ck.record(f"B2_runbook_bpb_{short}", float(run.loc[mname, "bpb"]))
    actual_dir = "higher" if run.loc[MAIN, "perplexity"] > run.loc["Qwen/Qwen3-0.6B-Base", "perplexity"] else "lower"
    print(f"Post-trained vs Base on the runbook: {actual_dir} perplexity (you predicted {PREDICT_POST_TRAINING or 'nothing'}).")
    ck.record("B2_post_training_direction", actual_dir)

## B3. More context, lower perplexity

The chapter's third rule of thumb: the more context a model has, the less uncertain it is about what comes next. This is the idea behind the next few weeks: in Labs 07 and 08 you'll give the help-desk assistant the right Larkspur documents as context, so it can predict (and answer) far better than from its training alone. Here the cell measures the perplexity of the **second half** of the runbook passage twice: on its own, and with the first half in front of it as context (only the second half is scored; that is what the `context=` argument of `lm_metrics()` does).

**TODO:** compute by how many **percent** the perplexity dropped when the first half was given as context.

In [ ]:
cut = PASSAGE.index("\n", len(PASSAGE) // 2)          # split at a line break near the middle
FIRST, SECOND = PASSAGE[:cut], PASSAGE[cut:]
ppl_alone = lm_metrics(tok, model, SECOND)["perplexity"]
ppl_with_context = lm_metrics(tok, model, SECOND, context=FIRST)["perplexity"]
print(f"second half alone: {ppl_alone:.2f}    with the first half as context: {ppl_with_context:.2f}")

In [ ]:
drop_pct = None     # TODO: percent decrease from ppl_alone to ppl_with_context (a positive number if it went down)
print(drop_pct)

In [ ]:
ck.check("B3_context_lowers_ppl", ppl_with_context < ppl_alone, points=1,
         hint="With the first half as context, the second half should be easier to predict. Re-run the cell above.")
ck.check("B3_drop_pct", drop_pct is not None and abs(drop_pct - (ppl_alone - ppl_with_context) / ppl_alone * 100) <= max(0.1, 0.01 * abs((ppl_alone - ppl_with_context) / ppl_alone * 100)), points=1,
         hint="Percent decrease = the difference divided by the starting value (ppl_alone), times 100.")
ck.record("B3_ppl_alone", ppl_alone)
ck.record("B3_ppl_with_context", ppl_with_context)

## B4. At work: flag abnormal tickets with perplexity

*Perplexity Interpretation and Use Cases* points out that perplexity is highest for unpredictable text, such as gibberish, so it **can be used to detect abnormal texts**. That is a practical job at an MSP: Larkspur's ticket table alone holds about 1,500 tickets that came in by email, portal, and phone, and before tickets are summarized by an AI or used to fine-tune a model (Labs 10-11), someone should catch the ones that are garbled by a broken email parser or are just a wall of pasted log output.

Below are seven ticket descriptions. Five are real tickets from Larkspur's ticket table; two are not ordinary. With seven tickets you can spot them by eye. With a year of tickets you can't, which is why a number helps. The scoring cell shows both perplexity and bits per byte, because B1 showed that per-token perplexity can be fooled by text that gets cut into many tiny tokens.

**TODO (before you run the scoring cell):** read the seven tickets and list the ones you expect to stand out.

In [ ]:
TICKETS = {
    'ticket_1': "Teams keeps signing me out and then it's stuck on the loading screen with the purple logo. I tried unplugging everything and plugging it back in. I have a meeting at 2 so if someone could look before then that would be great.",
    'ticket_2': "Printer says offline on my computer. Everyone else can print. Sorry to bug you guys again. I'm not very good with computers so please be patient with me lol. Also my mouse is kind of sticky but that's a different issue. Thank you",
    'ticket_3': "Night dispatch here. The dispatch system froze about 10 minutes ago and nobody can get back in. Screens just spin and then say the server is not responding. We have trucks rolling at 4 AM and I can't assign loads. Please call me back ASAP.",
    'ticket_4': '=?UTF-8?B?UHJpbnRlciBpcyBvZmZsaW5lIGFnYWlu?= Ã¢â‚¬Å“HiÃ¢â‚¬Â  ÃƒÂ¢ s3nt fr0m my ï¿½ï¿½ --_000_BN8PR12MB3417 Content-Transfer-Encoding: base64 PGh0bWw+PGJvZHk+PHA+VGhhbmtzIGFnYWluPC9wPjwvYm9keT48L2h0bWw+ ZXJyb3IgY29kZSAweDAwMDAwNzA5 ÃƒÂ©rr q8Zr7 Xw2kP0 vM9f ï¿½ =20 =3D=3D jfk2 0a8d',
    'ticket_5': "Thanks for getting us back up yesterday. Our owner wants to know why the backup internet didn't kick in on its own and what we're doing so it doesn't happen again. Can you send something in writing we can share? Also can we test the backup line on purpose some weekend?",
    'ticket_6': 'Pasted from Event Viewer:\nLog Name: System\nSource: Service Control Manager\nDate: 10/29/2026 8:14:02 AM\nEvent ID: 7031\nLevel: Error\nDescription: The Print Spooler service terminated unexpectedly. It has done this 3 time(s). The following corrective action will be taken in 60000 milliseconds: Restart the service.\nFaulting application name: spoolsv.exe, version: 10.0.22621.4249, faulting module name: PrintConfig.dll, exception code: 0xc0000005, fault offset: 0x000000000004a1f3, process id: 0x1f2c',
    'ticket_7': "We're getting a bunch of emails from a vendor we use with a 'new bank account' for payments. Doesn't feel right. This also happened last month and someone fixed it but I don't remember who. I'm out tomorrow so today would be ideal.",
}
for name, text in TICKETS.items():
    print(f"--- {name} ---\n{text}\n")

In [ ]:
MY_GUESS_ODD = []     # TODO: names of the tickets you expect to stand out, e.g. ["ticket_1"]

In [ ]:
b4 = pd.DataFrame([{"ticket": name, **lm_metrics(tok, model, text)} for name, text in TICKETS.items()])
b4 = b4.set_index("ticket").sort_values("bpb")
display(b4[["tokens", "perplexity", "bpb"]].round(2))

**Reading the table:** the tickets are sorted by bits per byte, from most predictable (top) to least (bottom). Compare the two columns: does perplexity separate the garbled ticket from the normal ones as clearly as bpb does? Short tickets have noisy per-token scores, and garbled text is cut into many tiny tokens (look at `tokens`), the same effect you saw with `random_chars` in B1.

**TODO:** choose a **bits-per-byte** threshold `FLAG_ABOVE`. Tickets above it get sent to a person to look at instead of going straight into the AI pipeline. Pick it so the garbled ticket is caught but normal requests from users are not. Whether the pasted log should be flagged is your call; explain it in your analysis.

In [ ]:
FLAG_ABOVE = None     # TODO: a bits-per-byte (bpb) value read from the table above
flagged = [] if FLAG_ABOVE is None else b4.index[b4["bpb"] > FLAG_ABOVE].tolist()
print("Threshold:", FLAG_ABOVE, "  Flagged:", flagged)

In [ ]:
ODD_GARBLED, ORDINARY = 'ticket_4', ['ticket_1', 'ticket_2', 'ticket_3', 'ticket_5', 'ticket_7']
ck.check("B4_guess", isinstance(MY_GUESS_ODD, list) and len(MY_GUESS_ODD) >= 1 and set(MY_GUESS_ODD) <= set(TICKETS),
         points=1, hint='List at least one ticket name, spelled as printed (for example "ticket_1").')
ck.check("B4_threshold_set", isinstance(FLAG_ABOVE, (int, float)), points=1, hint="Set FLAG_ABOVE to a number on the scale of the bpb column (not the perplexity column).")
ck.check("B4_garbled_flagged", ODD_GARBLED in flagged, points=2,
         hint="The garbled ticket is not above your threshold. Find it in the table and set FLAG_ABOVE below its bpb.")
ck.check("B4_ordinary_not_flagged", isinstance(FLAG_ABOVE, (int, float)) and not set(flagged) & set(ORDINARY), points=1,
         hint="Your threshold also flags normal user requests. Raise FLAG_ABOVE above the highest bpb of an ordinary ticket.")
ck.record("B4_flag_above", FLAG_ABOVE)
ck.record("B4_flagged", flagged)
for name in b4.index:
    ck.record(f"B4_ppl_{name}", float(b4.loc[name, "perplexity"]))
    ck.record(f"B4_bpb_{name}", float(b4.loc[name, "bpb"]))

# Written analysis

Use **your numbers** from this notebook and cite the chapter by section name.

**Q1. What perplexity can and can't tell an IT team.** (a) From B1: how did your predicted ranking compare with the actual one? Using your numbers, explain why the shuffled runbook scored so much worse than the original even though it has exactly the same words, and why *Pride and Prejudice* scored so low. (b) From B4: which bits-per-byte threshold did you choose, what did it flag, and did you flag the pasted log? Why would a threshold on perplexity alone have worked less well here? Would you use this as an automatic first filter on incoming Larkspur tickets? Name one problem it catches and one it can't catch (think about a ticket, or an assistant answer, that is fluent and well written but simply wrong). Cite *Perplexity Interpretation and Use Cases* (and, if you use the definitions, *Understanding Language Modeling Metrics*).

*Your answer:*

**Optional (not graded):**
- *Model and tokenizer (B2).* Why can't you compare SmolLM2's perplexity with Qwen's directly, while bits-per-byte is comparable? Did post-training raise or lower perplexity, and does that match the Warning box in *Perplexity Interpretation and Use Cases*?
- *Random characters.* How does your random-character BPB compare with 6.57 bits, the cost of a blind uniform guess among 95 symbols? Why can a model trained on English do even worse than blind guessing on random text?
- *Context (B3).* What does your perplexity drop suggest about giving the help-desk assistant the right documents in Labs 07-08?

# AI-use reflection

The AI-use reflection for Lab 04 is in the **cluster notebook** (one reflection for the whole lab). Include anything you used AI for in this Colab notebook there.

## Submit

Run this last, save (File > Download > .ipynb), upload the .ipynb.

In [ ]:
ck.report()